# 03: Etap 2, U-Net doszlifowujący (MotionNet + U-Net)

**Wejście:** wytrenowany MotionNet (etap 1) + zbiór z `01`. **Wynik:** `runs/<data>_refine_dual/best.pt` z wagami OBU sieci.

MotionNet jest zamrożony; uczy się tylko U-Net (+ mały moduł przesunięcia kamery B), który:
* poprawia lokalnie pola przesunięć dla A i B,
* dopasowuje globalne przesunięcie kamery B (odstęp kamer i synchronizacja),
* decyduje, z której kamery wziąć każdy piksel (brzegi, które wyszły poza kadr A, bierze z B),
* dodaje drobną poprawkę obrazu (ostrość, jasność).

In [ ]:
# === SETUP: uruchamiaj zawsze jako pierwszą komórkę ===
# Kod jest pobierany z GitHuba na maszynę Colaba (lokalne pliki z VS Code tam nie trafiają).
# Po zmianie kodu: commit + push w VS Code, potem ponownie ta komórka (robi git pull).
REPO = 'LOGIN/NAZWA_REPO'      # <- UZUPEŁNIJ RAZ: repozytorium na GitHubie
BRANCH = 'main'

import os, sys, subprocess, getpass, importlib
from google.colab import drive
drive.mount('/content/drive')

TOKEN_FILE = '/content/drive/MyDrive/rs_project/.github_token'   # opcjonalnie: token zapisany na Drive
CODE = '/content/code'

def _git(*args):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or r.stdout).replace(token, '***') if 'token' in globals() else r.stderr
        raise RuntimeError('git ' + args[0] + ' nie powiodło się:\n' + msg)
    return r.stdout

if not os.path.exists(CODE):
    token = (open(TOKEN_FILE).read().strip() if os.path.exists(TOKEN_FILE)
             else getpass.getpass('GitHub token (fine-grained, tylko odczyt): '))
    _git('clone', '-q', '-b', BRANCH, f'https://{token}@github.com/{REPO}.git', CODE)
else:
    _git('-C', CODE, 'pull', '-q')
os.chdir(CODE)
sys.path.insert(0, f'{CODE}/src')
import project_paths as P
importlib.reload(P)
P.ensure_dirs()
print(_git('log', '-1', '--format=kod: %h  %s  (%cd)', '--date=short'))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Co mamy na Drive? (wagi MotionNet i zbiór)

In [ ]:
P.migrate_old_layout()
P.status()
MOTION_CKPT = P.find_run('motion_dual')
print('\nMotionNet:', MOTION_CKPT)
assert MOTION_CKPT, 'Brak wag MotionNet - uruchom trening w 02_motionnet.ipynb'

In [ ]:
# Zbiór danych: rozpakowanie z Drive na dysk maszyny Colaba (szybki odczyt w trakcie treningu)
if not os.path.exists('/content/dataset/train'):
    z = P.find_dataset_zip()
    assert z, 'Brak zbioru na Drive - najpierw uruchom 01_dataset.ipynb'
    print('rozpakowuję', z, '...')
    !unzip -q -n "{z}" -d /content
print(len(os.listdir('/content/dataset/train')), 'próbek train /', len(os.listdir('/content/dataset/val')), 'val')
WORKERS = min(8, os.cpu_count())

## 2. Trening etapu 2
Na GPU z Colab Pro można zwiększyć `BS` do 32, jeśli starczy pamięci.
Po rozłączeniu uruchom ponownie SETUP i rozpakowanie, a tutaj ustaw `RESUME = True`.

In [ ]:
from datetime import date
RUN = f'{date.today()}_refine_dual'       # przy wznawianiu wpisz nazwę istniejącego runu
EPOCHS, BS = 15, 16
RESUME = False
resume = f'--resume "{P.RUNS / RUN / "last.pt"}"' if RESUME else ''
!python src/train.py --data /content/dataset --mode refine --motion-ckpt "{MOTION_CKPT}" \
    --epochs {EPOCHS} --bs {BS} --workers {WORKERS} --out "{P.RUNS}" --name {RUN} {resume}

## 3. Porównanie: sam MotionNet vs MotionNet + U-Net
* `psnr` / `ssim`: obszar widoczny w kamerze A (porównywalny z wynikami etapu 1)
* `psnr_b`: brzegi widoczne **tylko** w kamerze B (tu U-Net powinien dać największy zysk), `_in` = bez korekcji
* `epe`, `epe_b`: średni błąd wektorów przesunięć A i B [px]; `epe_in` = bez korekcji

In [ ]:
import json, pandas as pd
rows = []
for ck in [MOTION_CKPT, P.RUNS / RUN / 'best.pt']:
    out = ck.parent / 'eval.json'
    !python src/train.py --data /content/dataset --eval "{ck}" --eval-json "{out}" --workers {WORKERS} > /dev/null
    m = json.load(open(out))['metrics']
    rows.append({'run': ck.parent.name, **{k: round(v, 3) for k, v in m.items()}})
pd.DataFrame(rows).set_index('run')

In [ ]:
from IPython.display import Image, display
p = sorted((P.RUNS / RUN).glob('preview_ep*.png'))[-1]
print(p, '\nkolumny: RS A | RS B | wynik | GS cel | mapa widoczności (ziel. = A, nieb. = tylko B, czerw. = żadna)')
display(Image(str(p)))

## 4. Przykładowe wyniki (`infer.py`, ten sam skrypt działa lokalnie na CPU)
Wagi do testów lokalnych pobierz z Google Drive (`rs_project/runs/<RUN>/best.pt`):
w rozszerzeniu VS Code `files.download()` nie działa.

In [ ]:
OUTDIR = P.RESULTS / RUN
!python src/infer.py --ckpt "{P.RUNS / RUN / 'best.pt'}" --data /content/dataset/val --n 8 --out "{OUTDIR}"
for p in sorted(OUTDIR.glob('*.png'))[:4]:
    display(Image(str(p)))

## 5. Po treningu
Dopisz wiersz do `experiments/README.md` (run, data, zbiór, wyniki z tabeli powyżej) i zrób commit notatnika razem z wynikami.